# Working over the spectral curve $\DeclareMathOperator{\ord}{ord}\DeclareMathOperator{\Fr}{Fr}$

In [1]:
import sys
sys.path.insert(0,"..") # dalgebra is here
from dalgebra import *
from dalgebra.misc.quotient import *

## 1. Introduction and notation

This notebook tries to work over the spectral curve of a linear differential operator. Let us provide some definitions:

* Let $\mathbb{L}$ be a linear differential operator oer a field $\mathbb{K}$ of the form:
  $$\mathbb{L} = \partial^n + u_{n-2}\partial^{n-2} + \ldots + u_1\partial + u_0.$$
* We define $\mathcal{Z}(\mathbb{L}) = \{B\in\mathbb{K}[\partial]\ :\ [B,L] = 0\}$. We know that (Goodearl):
  $$\mathcal{Z}(\mathbb{L}) = C[\mathbb{L}] \oplus G_1 C[\mathbb{L}] \oplus \ldots \oplus G_d C[\mathbb{L}],$$
  where $d < n$, $\ord(G_i) < \ord(G_{i+1})$ and $\ord(G_i) \not\equiv \ord(G_j)\pmod{n}$.
* We know that $\mathcal{Z}(\mathbb{L}) \simeq C[\lambda,\mathbf{\mu}]/BC(\mathbb{L})$, where $BC(\mathbb{L})$ is the Burchnall-Chaundy ideal of $\mathbb{L}$, which is defined by
  $$BC(\mathbb{L}) = \{P(\lambda,\mu_1,\ldots,\mu_d)\in C[\lambda,\mathbf{\mu}]\ :\ P(L,G_1,\ldots,G_d) = 0\}.$$
* Let $D = C[\lambda,\mathbf{\mu}]/BC(\mathbb{L})$ be an integral domain and $\mathbb{F} = \Fr(D)$.

**Main goal of notebook**: see how to work over the differential field generated from $\mathbb{K}$ and $\mathbb{F}$.

## 2. Computing with an example

Let us start with a simple example using elliptic functions.

In [2]:
B = PolynomialRing(QQ, ("g_2", "g_3")) # base field of constants
DB = DifferentialRing(B) # add the differential structure
g_2, g_3 = DB.gens()

K = DElliptic(DB.fraction_field(), "eta_p^2 - eta^3 - g_2*eta - g_3", varname="eta")
DO = DifferentialPolynomialRing(K, "z") # linear differential operators
eta = K.gen(); eta_p = K.gen_p()
z = DO.gen("z")

## Creting the operator L
L = z[4] - 6*eta*z[2] - 3*eta_p*z[1] - 41/40*g_2*z[0]

In [3]:
%%time
from dalgebra.commutators import *
## COMPUTING THE CENTRALIZER OF THIS OPERATOR
Us = tuple(L.coefficient_full(z[i]) for i in range(L.order(z) - 1))
_, (_,G_1,G_2,G_3),_ = GetCentralizer(Us, 10, starting_level=5, ignore_bound=True)

## COMPUTING THE BURCHNALL-CHAUNDY IDEAL
I = BC_ideal(L, [G_1,G_2,G_3], z)

CPU times: user 8.57 s, sys: 148 ms, total: 8.72 s
Wall time: 8.82 s


In [4]:
for el in I.basis:
    show(el)

mu_1^2 - mu_3

mu_1*mu_2 - lambda_^3 + (87*g_2)/40*lambda_^2 + (1377*g_2^2)/1600*lambda_ + (-158711*g_2^3)/64000

mu_1*mu_3 - mu_2*lambda_^2 + (16*g_2)/5*mu_2*lambda_ + (-3871*g_2^2)/1600*mu_2 + (-27*g_3)/4*mu_1*lambda_ + (2133*g_2*g_3)/160*mu_1

mu_2^2 - mu_3*lambda_ + (-41*g_2)/40*mu_3 + (27*g_3)/4*lambda_^2 + (-513*g_2*g_3)/80*lambda_ + (-87453*g_2^2*g_3)/6400

mu_2*mu_3 - mu_1*lambda_^3 + (87*g_2)/40*mu_1*lambda_^2 + (1377*g_2^2)/1600*mu_1*lambda_ + (-158711*g_2^3)/64000*mu_1

mu_3^2 + (-27*g_3)/4*mu_3*lambda_ + (2133*g_2*g_3)/160*mu_3 - lambda_^5 + (43*g_2)/8*lambda_^4 + (-1363*g_2^2)/160*lambda_^3 + (181*g_2^3)/6400*lambda_^2 + (205163*g_2^4)/20480*lambda_ + (-614370281*g_2^5)/102400000

In [5]:
D = DifferentialRing(QuotientRing_extended(I.ring(), I, names=None))
F = D.fraction_field()

In [6]:
K_F = K.change_base(F)

In [7]:
## Changing into the field with the quotient
mu_1, mu_2, mu_3, _lambda = K_F.base().gens()
eta, eta_p = K_F(eta), K_F(eta_p)

In [8]:
DO_F = DO.change_ring(K_F)

In [9]:
L_lambda = DO_F(L) - DO_F(_lambda)*z[0]

In [10]:
G1_mu = DO_F(G_1) - DO_F(mu_1)*z[0]
G2_mu = DO_F(G_2) - DO_F(mu_2)*z[0]
G3_mu = DO_F(G_3[1]) - DO_F(mu_3)*z[0]

In [11]:
%%time
G1_mu_ders = [G1_mu.derivative(times=i) for i in range(L_lambda.order(z))]
L_lambda_ders = [L_lambda.derivative(times=i) for i in range(G1_mu.order(z))]
rows_G1 = [[el.coefficient_full(z[i]) for i in range(L_lambda.order(z)+G1_mu.order(z))] for el in G1_mu_ders]
rows_L = [[el.coefficient_full(z[i]) for i in range(L_lambda.order(z)+G1_mu.order(z))] for el in L_lambda_ders]

CPU times: user 12.6 s, sys: 8.72 ms, total: 12.6 s
Wall time: 12.5 s


IOStream.flush timed out
IOStream.flush timed out
